In [4]:

import pandas as pd
import math

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)



df = pd.read_csv("mushrooms.csv")

print("========================================")
print("DATASET INFORMATION")
print("========================================")

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())


if "Id" in df.columns:
    df = df.drop(columns=["Id"])

train_df, test_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["class"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\n========================================")
print("TRAIN / TEST SPLIT")
print("========================================")

print("Training samples:", len(train_df))
print("Testing samples :", len(test_df))



def foil_gain(p0, n0, p1, n1):

    if p1 == 0:
        return -float("inf")

    if n1 == 0:
        return float("inf")

    probability_before = p0 / (p0 + n0)

    probability_after = p1 / (p1 + n1)

    gain = p1 * (
        math.log2(probability_after)
        - math.log2(probability_before)
    )

    return gain

def generate_candidates(data, used_features):

    candidates = []

    for feature in data.columns:

        if feature == "class":
            continue

        if feature in used_features:
            continue

        values = data[feature].unique()

        for value in values:

            candidates.append(
                (feature, value)
            )

    return candidates


def print_rule(rule):

    conditions = []

    for feature, value in rule:

        conditions.append(
            f"{feature} = {value}"
        )

    if len(conditions) == 0:

        print("IF TRUE THEN class = p")

    else:

        print(
            "IF "
            + " AND ".join(conditions)
            + " THEN class = p"
        )



def learn_one_rule(pos, neg, data):

    rule = []

    current_pos = pos.copy()
    current_neg = neg.copy()

    used_features = set()

    print("\n----------------------------------------")
    print("Learning a New Rule")
    print("----------------------------------------")

    while len(current_neg) > 0:

        p0 = len(current_pos)
        n0 = len(current_neg)

        candidates = generate_candidates(
            data,
            used_features
        )

        if len(candidates) == 0:
            break

        best_literal = None
        best_gain = -float("inf")

        best_pos = None
        best_neg = None


        for literal in candidates:

            feature, value = literal

            new_pos = current_pos[
                current_pos[feature] == value
            ]

            new_neg = current_neg[
                current_neg[feature] == value
            ]
            p1 = len(new_pos)
            n1 = len(new_neg)

            gain = foil_gain(
                p0,
                n0,
                p1,
                n1
            )

            if gain > best_gain:

                best_gain = gain

                best_literal = literal

                best_pos = new_pos
                best_neg = new_neg

        if best_literal is None:
            break


        rule.append(best_literal)

        feature, value = best_literal

        used_features.add(feature)


        current_pos = best_pos
        current_neg = best_neg

        print(
            "Selected Literal:",
            feature,
            "=",
            value
        )

        if best_gain == float("inf"):
            print("FOIL Gain: Infinite")
        else:
            print(
                "FOIL Gain:",
                round(best_gain, 4)
            )

        print(
            "Positive examples:",
            len(current_pos)
        )

        print(
            "Negative examples:",
            len(current_neg)
        )


        if len(current_neg) == 0:
            break

    return rule



def foil_train(data, max_rules=20):

    positives = data[
        data["class"] == "p"
    ].copy()


    negatives = data[
        data["class"] == "e"
    ].copy()

    # List of learned rules
    learned_rules = []

    print("\n========================================")
    print("STARTING FOIL ALGORITHM")
    print("========================================")

    print(
        "Initial Positive Examples:",
        len(positives)
    )

    print(
        "Initial Negative Examples:",
        len(negatives)
    )

    while len(positives) > 0:

        # Limit number of rules
        if len(learned_rules) >= max_rules:

            print("\nMaximum number of rules reached.")

            break


        new_rule = learn_one_rule(
            positives,
            negatives,
            data
        )

        if len(new_rule) == 0:

            print(
                "\nNo more useful rules can be generated."
            )

            break


        learned_rules.append(new_rule)

        covered = positives.copy()

        for feature, value in new_rule:

            covered = covered[
                covered[feature] == value
            ]


        print("\nLearned Rule:")

        print_rule(new_rule)

        print(
            "Positive examples covered:",
            len(covered)
        )

        positives = positives.drop(
            covered.index
        )

        print(
            "Positive examples remaining:",
            len(positives)
        )

    return learned_rules



rules = foil_train(
    train_df,
    max_rules=20
)



print("\n\n========================================")
print("FINAL LEARNED RULES")
print("========================================")

if len(rules) == 0:

    print("No rules were learned.")

else:

    for i, rule in enumerate(rules, start=1):

        print(f"\nRule {i}:")

        print_rule(rule)




def predict_row(row, rules):


    for rule in rules:

        rule_matches = True


        for feature, value in rule:

            if row[feature] != value:

                rule_matches = False

                break


        if rule_matches:

            return "p"

    return "e"

predictions = []

for _, row in test_df.iterrows():

    prediction = predict_row(
        row,
        rules
    )

    predictions.append(prediction)



y_true = test_df["class"]

accuracy = accuracy_score(
    y_true,
    predictions
)


print("\n\n========================================")
print("FOIL ALGORITHM RESULTS")
print("========================================")

print(
    "Training Samples:",
    len(train_df)
)

print(
    "Testing Samples:",
    len(test_df)
)

print(
    "Number of Rules:",
    len(rules)
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)



print("\n========================================")
print("CLASSIFICATION REPORT")
print("========================================")

print(
    classification_report(
        y_true,
        predictions,
        labels=["e", "p"],
        target_names=[
            "Edible",
            "Poisonous"
        ],
        zero_division=0
    )
)



print("\n========================================")
print("CONFUSION MATRIX")
print("========================================")

cm = confusion_matrix(
    y_true,
    predictions,
    labels=["e", "p"]
)

print(cm)


print("\n========================================")
print("SUMMARY")
print("========================================")

print("FOIL learned", len(rules), "rules.")

print(
    "The rules classify mushrooms as poisonous (p)"
    " or edible (e)."
)

print(
    "The best literals were selected using"
    " FOIL Information Gain."
)

DATASET INFORMATION
Dataset Shape: (8124, 23)

First 5 rows:
  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                  